In [0]:
table_name = "fact_procedures"
silver_table_fqn = f"hdfc_ergo.hdfc_silver.{table_name}"

df = spark.read.table("hdfc_ergo.hdfc_bronze.fact_procedures")

In [0]:
from pyspark.sql.functions import col, upper, trim, when, to_date, to_timestamp, row_number, abs as spark_abs
from pyspark.sql.window import Window

# 1. Enrich: Map Source SKs -> Business Keys (Including Claim_SK -> Claim_ID)
df_map_claim = spark.read.table("hdfc_ergo.hdfc_bronze.fact_claims").select(col("claim_sk").cast("int").alias("src_claim_sk"), col("claim_id"))
df_map_member = spark.read.table("hdfc_ergo.hdfc_bronze.dim_members").select(col("member_sk").cast("int").alias("src_member_sk"), col("member_id"))
df_map_provider = spark.read.table("hdfc_ergo.hdfc_bronze.dim_providers").select(col("provider_sk").cast("int").alias("src_provider_sk"), col("provider_id"))
df_map_facility = spark.read.table("hdfc_ergo.hdfc_bronze.dim_facilities").select(col("facility_sk").cast("int").alias("src_facility_sk"), col("facility_id"))
df_map_cpt = spark.read.table("hdfc_ergo.hdfc_bronze.dim_cpt_codes").select(col("cpt_code_sk").cast("int").alias("src_cpt_code_sk"), col("cpt_code"))
df_map_surgeon = spark.read.table("hdfc_ergo.hdfc_bronze.dim_providers").select(col("provider_sk").cast("int").alias("src_surgeon_sk"), col("provider_id").alias("surgeon_id"))
df_map_asst_surgeon = spark.read.table("hdfc_ergo.hdfc_bronze.dim_providers").select(col("provider_sk").cast("int").alias("src_assistant_surgeon_sk"), col("provider_id").alias("assistant_surgeon_id"))

df = df.join(df_map_claim, df["claim_sk"] == df_map_claim["src_claim_sk"], "left").drop("src_claim_sk", "claim_sk")
df = df.join(df_map_member, df["member_sk"] == df_map_member["src_member_sk"], "left").drop("src_member_sk", "member_sk")
df = df.join(df_map_provider, df["provider_sk"] == df_map_provider["src_provider_sk"], "left").drop("src_provider_sk", "provider_sk")
df = df.join(df_map_facility, df["facility_sk"] == df_map_facility["src_facility_sk"], "left").drop("src_facility_sk", "facility_sk")
df = df.join(df_map_cpt, df["cpt_code_sk"] == df_map_cpt["src_cpt_code_sk"], "left").drop("src_cpt_code_sk", "cpt_code_sk")
df = df.join(df_map_surgeon, df["surgeon_sk"] == df_map_surgeon["src_surgeon_sk"], "left").drop("src_surgeon_sk", "surgeon_sk")
df = df.join(df_map_asst_surgeon, df["assistant_surgeon_sk"] == df_map_asst_surgeon["src_assistant_surgeon_sk"], "left").drop("src_assistant_surgeon_sk", "assistant_surgeon_sk")

# 2. Standardize, Cast Dates/Timestamps/Ints/Decimals/Booleans (Condensed for brevity)
for c in ["procedure_id", "claim_id", "member_id", "provider_id", "facility_id", "cpt_code", "surgeon_id", "assistant_surgeon_id", "anesthesia_type", "complication_type", "outcome_status"]:
    df = df.withColumn(c, upper(trim(col(c))))

df = df.withColumn("procedure_date", to_date(trim(col("procedure_date"))))
for ts_col in ["procedure_start_time", "procedure_end_time", "record_created_date", "record_modified_date"]:
    df = df.withColumn(ts_col, to_timestamp(trim(col(ts_col))))

df = df.withColumn("procedure_duration_minutes", trim(col("procedure_duration_minutes")).cast("int"))

for dec_col in ["facility_charge", "surgeon_charge", "anesthesia_charge", "total_procedure_charge", "insurance_paid"]:
    is_valid = trim(col(dec_col)).rlike("^[+-]?([0-9]+\\.?[0-9]*|[0-9]*\\.?[0-9]+)$")
    df = df.withColumn(dec_col, when(is_valid, trim(col(dec_col)).cast("decimal(15,2)")).otherwise(None))
    df = df.withColumn(dec_col, spark_abs(col(dec_col)))

for bool_col in ["complication", "readmission_required", "emergency_procedure", "elective_procedure", "laparoscopic"]:
    df = df.withColumn(bool_col, when(upper(trim(col(bool_col))).isin("YES", "1", "TRUE"), True).when(upper(trim(col(bool_col))).isin("NO", "0", "FALSE"), False).otherwise(None).cast("boolean"))

dedup_window = Window.partitionBy("procedure_id").orderBy(col("_ingested_at").desc())
df = df.withColumn("_dq_is_deduped", row_number().over(dedup_window)).filter(col("_dq_is_deduped") == 1).drop("_dq_is_deduped", "_ingested_at", "_source_file", "procedure_sk")

display(df.limit(5))

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table_fqn)
print(f"✅ Successfully wrote {table_name} to Silver layer.")